# **Components in LangChain**

* Models
* Prompts
* Chains
* Indexes (doc -oader, text-splitter, vectorstore, retriever)
* Agents

In [1]:
import langchain

print(langchain.__version__)

1.3.9


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
def model_setup():
    from dotenv import load_dotenv
    load_dotenv()

    from langchain_google_genai import ChatGoogleGenerativeAI
    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

    return model

## **MODELS**

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [7]:
response = model.invoke("Hey, tell me a joke.")
response.content

"Why don't scientists trust atoms?\n\nBecause they make up everything!"

#### Embedding models

In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [10]:
embedding_mdoel = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [15]:
result = embedding_mdoel.embed_query("Hey, tell me a joke.")    # embed_documents() method can be used to embed multiple texts at once
type(result), type(result[0])

(list, float)

## **PROMPTS**

* basic validation
* reusable
* coupled with langchain ecosystem

In [15]:
from langchain_core.prompts import PromptTemplate, load_prompt

In [18]:
template = PromptTemplate(template= "hello my name is {name}", input_variables=["name"], validate_template=True)
template

PromptTemplate(input_variables=['name'], input_types={}, partial_variables={}, template='hello my name is {name}', validate_template=True)

In [20]:
prompt = template.invoke({"name": "Aman"})
prompt

StringPromptValue(text='hello my name is Aman')

In [13]:
response = model.invoke(prompt)
response.content

"Hello Aman, it's nice to meet you! I'm an AI assistant.\n\nHow can I help you today?"

In [14]:
template.save("template.json")

C:\Users\aman\AppData\Local\Temp\ipykernel_16484\580953154.py:1: LangChainDeprecationWarning: The method `BasePromptTemplate.save` was deprecated in langchain-core 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  template.save("template.json")


In [21]:
template = load_prompt("template.json")

### Different type of messages
- System message
- Human message
- AI message

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [8]:
messages = [ 
    SystemMessage("You are a professional comedian"),
    HumanMessage("Tell me an joke")
]

In [10]:
messages.append(AIMessage(model.invoke(messages).content))
messages

[SystemMessage(content='You are a professional comedian', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Tell me an joke', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Alright, alright, settle down folks, you\'re too kind!\n\nI was on the phone with my internet provider the other day, you know the drill. "Press 1 for billing, 2 for technical support, 3 for sales..." I swear, I pressed so many numbers, I felt like I was playing telephone bingo.\n\nAfter about fifteen minutes, I finally got to the option for "speak to a representative." I hit it, hopeful.\n\nAnd the automated voice says, "All representatives are currently assisting other customers. Your approximate wait time is one hour and forty-five minutes."\n\nOne hour and forty-five minutes! For a *representative*! I hung up.\n\nI swear, at this point, I think "customer service representatives" are just like Bigfoot. Everyone\'s heard of them, some people claim to have seen one, but nobody ca

### Chat Prompt Template

In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [20]:
chat_template = ChatPromptTemplate([
    ("system", "You are a professional {role}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Describe what you do in brief")],
)

prompt = chat_template.invoke({"role": "comedian", "chat_history": [HumanMessage(content="Hello")]})
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional comedian', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}), HumanMessage(content='Describe what you do in brief', additional_kwargs={}, response_metadata={})])

## **STRUCTURED OUTPUT**

using:
- TypedDIct
- Pydantic
- json schema

In [4]:
from typing import TypedDict, Annotated, Optional   #, Literal

In [27]:
model = model_setup()

In [5]:
class Person(TypedDict):
    name: Annotated[str, "Name of the person"]
    role: str
    age: int

In [17]:
model = model.with_structured_output(Person)

In [ ]:
model.invoke("i am aman and i am currently a student and my age is 22")

{'name': 'aman', 'role': 'student', 'age': 22}

In [18]:
from pydantic import BaseModel, EmailStr, Field

In [7]:
class student(BaseModel):       # pydantic also does type coercing too
    name: str
    age: Optional[int] = Field(gt=0, default=18)
    email: Optional[EmailStr] = None

In [8]:
s1 = student.model_validate({"name": "aman"})

In [23]:
model = model.with_structured_output(student, method="json_output")

In [24]:
model.invoke("i am aman and i am currently a student and my age is 22")

student(name='aman', age=22, email=None)

In [25]:
{
    "title":"student",
    "description":"student schema",
    "type": "object",
    "properties":{
        "name": {"type":"string"},
        "age": "integer",
    },
    "required":  ["name"],
}

{'title': 'student',
 'description': 'student schema',
 'type': 'object',
 'properties': {'name': {'type': 'string'}, 'age': 'integer'},
 'required': ['name']}

### Output parsers

- string output parser

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [38]:
template1 = PromptTemplate(template="write a detailed report (1 page) on {topic}", input_variables=["topic"])
template2 = PromptTemplate(template="write a short summary (5 lines) of the following text: \n{text}", input_variables=['text'])

In [4]:
model = model_setup()
parser = StrOutputParser()

In [42]:
chain = template1 | model | parser | template2 | model | parser

In [43]:
chain.invoke({"topic": "RAG"})

'Retrieval-Augmented Generation (RAG) is an architectural pattern that enhances Large Language Models (LLMs) by providing them dynamic access to external, up-to-date, and factual knowledge. It addresses inherent LLM limitations like knowledge cutoffs, hallucinations, and lack of transparency by grounding responses in verified data.\n\nThe RAG process involves indexing external documents into a vector database, then retrieving relevant information at runtime to augment user queries for the LLM. This significantly improves accuracy, factuality, and explainability by enabling source citation, proving more cost-effective than continuous LLM fine-tuning.\n\nWhile facing challenges such as retrieval quality and scalability, RAG is a transformative solution. It empowers LLMs to be more reliable and adaptable, with widespread applications from customer support to enterprise knowledge management.'

- json output parser

In [44]:
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
parser = JsonOutputParser()
template = PromptTemplate(template="give me the name, job and city of a medival fictional character.\n{format_instruction}",
                          input_variables=[],
                          partial_variables={"format_instruction": parser.get_format_instructions()})

In [54]:
chain = template | model | parser

In [57]:
chain.invoke({})

{'name': 'Elara', 'job': 'Guild Scribe', 'city': 'Veridian'}

- structured output parser

In [58]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

In [59]:
schema = [
    ResponseSchema(name="myth", description="myth about the given topic"),
    ResponseSchema(name="actual_fact", description="actual fact instead of the myth about the given topic"),
]

In [60]:
parser = StructuredOutputParser.from_response_schemas(schema)

In [ ]:
template = PromptTemplate(
    template="tell me an myth and the actual fact about {topic}\n{format_instruction}",
    input_variables=["topic"],
    partial_variables={"format_instruction": parser.get_format_instructions()}
    )

In [62]:
chain = template | model | parser

In [63]:
chain.invoke({"topic": "Blackhole"})

{'myth': 'Black holes are cosmic vacuum cleaners that roam the universe, actively sucking up stars, planets, and everything else in their path.',
 'actual_fact': "Black holes only pull things in if they are gravitationally bound to them or if they get too close, crossing the 'event horizon'. Their gravitational pull, at a given distance, is no stronger than that of any other object of the same mass. For example, if our Sun were to instantly turn into a black hole of the same mass, Earth would continue to orbit it exactly as it does now, because the black hole's gravity at Earth's distance would be identical to the Sun's."}

- pydantic output parser

In [5]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [6]:
class Person(BaseModel):
    name: str = Field(description="name of the fictional charactor")
    age: int = Field(gt=18, description="age of the fictional character")
    city: str = Field(description="name of the city form which the person belongs to")

In [7]:
parser = PydanticOutputParser(pydantic_object=Person)
template = PromptTemplate(template="give me the name, job and city of a medival fictional character.\n{format_instruction}",
                          input_variables=[],
                          partial_variables={"format_instruction": parser.get_format_instructions()}
                          )

In [8]:
chain = template | model | parser

In [74]:
chain.invoke({})

Person(name='Sir Kaelan', age=32, city='Valoria')

## **CHAINS**

#### Sequential chains

In [9]:
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Person |         
        +--------+         


#### Parallel chains

In [22]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [23]:
prompt1 = PromptTemplate(
    template="write a short note on {topic}",
    input_variables=["topic"]
)

prompt2 = PromptTemplate(
    template="write 2 short question and answers on {topic}",
    input_variables=["topic"]
)

prompt3 = PromptTemplate(
    template="Merge the given short note and quiz questions and answers into one document\n\nshort note: {notes}\n\nquiz: {quiz}",
    input_variables=["notes", "quiz"]
)

In [24]:
parser = StrOutputParser()
model = model_setup()

In [25]:
parallel_chain = RunnableParallel({
    "notes": prompt1 | model | parser,
    "quiz": prompt2 | model | parser
})

merged_chain = prompt3 | model | parser

chain = parallel_chain | merged_chain

In [26]:
chain.get_graph().print_ascii()

                    +---------------------------+                      
                    | Parallel<notes,quiz>Input |                      
                    +---------------------------+                      
                       ***                   ***                       
                   ****                         ****                   
                 **                                 **                 
    +----------------+                          +----------------+     
    | PromptTemplate |                          | PromptTemplate |     
    +----------------+                          +----------------+     
             *                                           *             
             *                                           *             
             *                                           *             
+------------------------+                  +------------------------+ 
| ChatGoogleGenerativeAI |                  | ChatGoogleGenerati

In [27]:
chain.invoke({"topic": "cricket"})

"Here's the merged document combining the short note and quiz questions and answers:\n\n**Cricket: A Bat-and-Ball Game**\n\nCricket is a bat-and-ball game played between two teams of eleven players each. The objective is to score runs by hitting a ball bowled by a player from the opposing team and then running between two sets of wickets. The team that scores more runs wins. It's a sport rich in tradition, strategy, and skill, enjoyed by millions worldwide, particularly in countries like England, India, Australia, and the West Indies.\n\n---\n\n**Cricket Quiz**\n\nHere are two short cricket questions and answers:\n\n**Q1:** What is the term for a delivery that bounces before reaching the batsman?\n**A1:** A **bouncer**.\n\n**Q2:** How many balls are in an over in a standard game of cricket?\n**A2:** **Six** balls."

#### Conditional chains

In [47]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [48]:
class feedback(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="sentiment of the feedback")

In [49]:
model = model_setup()
parser1 = PydanticOutputParser(pydantic_object=feedback)
parser2 = StrOutputParser()

In [65]:
prompt1 = PromptTemplate(
    template="classify this given feed back as positive or negative: {feedback}\n{format_instructions}",
    input_variables=["feedback"],
    partial_variables={"format_instructions":parser1.get_format_instructions()}
)

prompt2 = PromptTemplate(
    template="write a short thank you reply for the {sentiment} feedback",
    input_variables=["sentiment"]
)

prompt3 = PromptTemplate(
    template="write an short apology reply for the {sentiment} feedback",
    input_variables=["sentiment"]
)

In [66]:
classifier_chain = prompt1 | model | parser1
brached_chain = RunnableBranch(
    (lambda x:x.sentiment == "positive", prompt2 | model | parser2),
    (lambda x:x.sentiment == "negative", prompt3 | model | parser2),
    RunnableLambda(lambda x: "could not find the sentiment"),
)
chain = classifier_chain | brached_chain

In [69]:
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | BranchOutput |      
     +--------------+      


In [68]:
chain.invoke({"feedback": "this was a really good restaurant"})

"Here are a few short thank you replies for positive feedback, choose the one that best fits your style:\n\n**Simple & Sweet:**\n\n*   Thank you so much! I really appreciate your kind words.\n*   That's wonderful to hear! Thank you for the positive feedback.\n*   Thanks a bunch! Glad you enjoyed it.\n*   So happy to hear that! Thank you.\n\n**Slightly More Detailed:**\n\n*   Thank you for the fantastic feedback! It means a lot.\n*   I'm so glad you feel that way! Thank you for your positive words.\n*   Your feedback is greatly appreciated! Thank you.\n*   Thanks for your wonderful feedback! It's encouraging to know.\n\n**Enthusiastic:**\n\n*   Wow, thank you! That's amazing to hear!\n*   Thank you for the incredibly positive feedback! So thrilled.\n\n**When you want to acknowledge their specific input (if applicable):**\n\n*   Thank you! I'm so glad [mention what they liked] worked out well for you.\n*   Thanks for the great feedback! I'm happy to hear you found [specific aspect] helpf

### Runnables

- RunnableSequence

In [71]:
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

In [78]:
model = model_setup()
parser = StrOutputParser()
prompt1 = PromptTemplate(template="tell me a joke")
prompt2 = PromptTemplate(template="Explain the given joke: {joke}", input_variables=["joke"])

In [79]:
chain = RunnableSequence(prompt, model, parser)

In [80]:
chain.invoke({})

'Why did the scarecrow win an award?\n\nBecause he was outstanding in his field!'

- RunnableParallel: used above (it sreturn a dict containing the outputs of all the parrallel chain)

- RunnablePassthrough: returns whatever it is given

In [81]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [87]:
parallel_chain = RunnableParallel({
    "joke": RunnablePassthrough(),
    "explanation": RunnableSequence(prompt2, model, parser)
})

In [88]:
chain = RunnableSequence(prompt1, model, parser, parallel_chain)

In [89]:
chain.invoke({})

{'joke': "Why don't scientists trust atoms?\n\nBecause they make up everything!",
 'explanation': 'This is a classic pun! Here\'s the breakdown:\n\n*   **"Atoms make up everything"**: This is a literal scientific fact. Atoms are the fundamental building blocks of all matter in the universe. Everything we can see, touch, and interact with is composed of atoms.\n\n*   **"Make up" as in "fabricate" or "lie"**: This is the double meaning that creates the humor. In everyday language, when someone "makes up" something, they are fabricating a story, lying, or inventing something that isn\'t true.\n\n**The Joke\'s Humor:**\n\nThe joke plays on the ambiguity of the phrase "make up."\n\n*   **The Setup:** "Why don\'t scientists trust atoms?" This creates an expectation that there\'s a reason for mistrust, perhaps something complicated or a scientific quirk.\n\n*   **The Punchline:** "Because they make up everything!"\n\n    *   **Literal interpretation (scientific):** Atoms are responsible for t

- RunnableLambda: convert a python function into a runnable

In [90]:
from langchain_core.runnables import RunnableLambda

In [ ]:
def word_count(text):
    return len(text.split())

runnable_word_counter = RunnableLambda(word_count)

RunnableLambda(word_count)

In [93]:
chain = RunnableSequence(prompt1, model, parser, runnable_word_counter)

In [94]:
chain.invoke({})

10

- RunnableBranch: conditional chains (used above)

## **DOCUMENT LOADERS**

- Text loader

In [1]:
from langchain_community.document_loaders import TextLoader

In [2]:
loader = TextLoader("text.txt", encoding="utf-8")

In [3]:
doc = loader.load()

In [4]:
doc, type(doc), type(doc[0])

([Document(metadata={'source': 'text.txt'}, page_content='why is this file even here?\n\nMaybe to test the text loader of langchain')],
 list,
 langchain_core.documents.base.Document)

- PyPDFLoader: it works page-wise, returns a langchain document for each page in the padf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader    # works the same ways as other loaders, (dont have a pdf to test it)

- Directory loader

In [6]:
from langchain_community.document_loaders import DirectoryLoader

In [ ]:
laoder = DirectoryLoader(
    path="path to folder/directory",
    glob="regular excpression for what files to laod",
    loader_cls="what type to laoder to use eg: PyPDFLoader",
)

lazy loading: it does not laod all the doucments in memory, it return a document generator object which loades docs in memory on demand

In [7]:
doc_generator_object = loader.lazy_load() 

- Web based laoder: used beautifulsoup and requests libraries, works well with static web pages, for dynamic web pages use `SeleniumURLLoader`

In [8]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [9]:
laoder = WebBaseLoader("url or list of urls")